# Plot spot position as time series data
env : data_vis_32

# 1.0 Import relevant packages

In [25]:
# import pypyodbc
import pandas as pd
import plotly.express as px
import seaborn as sns
from matplotlib.colors import to_hex

# 2.0 Import spot position and size QA data

In [26]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx"

df = pd.read_excel(data_path)

df.head(2)



,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,hor_rt_gradient,hor_lt_gradient,hor_fwhm,vert_rt_gradient,vert_lt_gradient,vert_fwhm,bltr_rt_gradient,bltr_lt_gradient,bltr_fwhm,tlbr_rt_gradient,tlbr_lt_gradient,tlbr_fwhm
0,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Centre,-0.2357,124.9224,-9.059840,9.157258,13.342140,-8.964427,9.333333,13.536155,-8.485281,8.747554,14.216962,-8.909545,8.992812,13.842831
1,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Left,-125.0153,125.4501,-8.871094,9.120482,13.562307,-8.964427,9.333333,13.712522,-8.747554,8.591347,14.310494,-8.747554,8.992812,13.842831


# 3.0 exploratory data analysis - understand your data

In [27]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41172 entries, 0 to 41171
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ADate             41172 non-null  datetime64[ns]
 1   MachineName       41172 non-null  object        
 2   Energy            41172 non-null  int64         
 3   Device            41172 non-null  object        
 4   Gantry Angle      41172 non-null  int64         
 5   Spot              41172 non-null  object        
 6   x-pos             41172 non-null  float64       
 7   y-pos             41172 non-null  float64       
 8   hor_rt_gradient   41172 non-null  float64       
 9   hor_lt_gradient   41172 non-null  float64       
 10  hor_fwhm          41172 non-null  float64       
 11  vert_rt_gradient  41172 non-null  float64       
 12  vert_lt_gradient  41172 non-null  float64       
 13  vert_fwhm         41172 non-null  float64       
 14  bltr_rt_gradient  4117

In [28]:
df.value_counts("MachineName"), df.value_counts("Device"), df.value_counts("Energy")

(MachineName
 Gantry 3    10576
 Gantry 1    10472
 Gantry 4    10357
 Gantry 2     9767
 Name: count, dtype: int64,
 Device
 XRV-3000    31815
 XRV-4000     9357
 Name: count, dtype: int64,
 Energy
 150    8250
 240    8243
 200    8233
 100    8232
 70     8214
 Name: count, dtype: int64)

# 4.0 filtering data

## calculate abs shift

In [29]:
sub_df = df[["ADate", "MachineName", "Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

## calculate expected spot positions
pred_xrv4000 = {
    'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175],
    'Top-Left': [-125, -125], 'Top-Centre': [0, -125], 'Top-Right': [125, -125],
    'Left': [-125, 0], 'Centre': [0, 0], 'Right': [125, 0],
    'Bottom-Left': [-125, 125], 'Bottom-Centre': [0, 125], 'Bottom-Right': [125, 125],
    'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]
}

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

# first calculate absolute shift
sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

# convert absolute shift to relative shift wrt Centre spot
group_cols = ["ADate", "MachineName", "Device", "Energy", "Gantry Angle"]

centre_shift = (
    sub_df[sub_df["Spot"] == "Centre"]
    .groupby(group_cols, as_index=False)[["abs_xpos", "abs_ypos"]]
    .mean()
    .rename(columns={"abs_xpos": "centre_abs_x", "abs_ypos": "centre_abs_y"})
)

sub_df = sub_df.merge(centre_shift, on=group_cols, how="left")

# create relative shift columns
sub_df["rel_xpos"] = sub_df["abs_xpos"] - sub_df["centre_abs_x"]
sub_df["rel_ypos"] = sub_df["abs_ypos"] - sub_df["centre_abs_y"]

In [30]:
print(sub_df.head(2))

                ADate MachineName  Energy    Device  Gantry Angle  \
0 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   
1 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   

            Spot     x-pos     y-pos  px_pos  py_pos  abs_xpos  abs_ypos  \
0  Bottom-Centre   -0.2357  124.9224       0     125   -0.2357   -0.0776   
1    Bottom-Left -125.0153  125.4501    -125     125   -0.0153    0.4501   

   centre_abs_x  centre_abs_y  rel_xpos  rel_ypos  
0       -0.5678       -0.0179    0.3321   -0.0597  
1       -0.5678       -0.0179    0.5525    0.4680  


In [31]:
def plotly_spot_position(df, pos, gantry, device, energy, gantry_angle, n_months, exclude_centre=False):
    """ plot spot position time series data
        df = dataframe
        gantry = "Gantry 1", "Gantry 2",
        pos = "rel_xpos" or "rel_ypos"
        device = "XRV-3000", "XRV-4000"
        energy = int
        gantry_angle = 0,90,180,270
        n_months = int
     """

    # only show data from last n months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df['ADate'] >= start_date) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle)
    ]

    if exclude_centre:
        selected_df = selected_df[selected_df["Spot"] != "Centre"]

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=pos,
        symbol='Spot',
        color='Spot',
        color_discrete_sequence=px.colors.qualitative.T10,
        title=f'{gantry} - relative shift wrt Centre - {pos}',
        labels={'ADate': 'Date', pos: 'Relative shift (mm)'},
        height=500
    )

    # Add tolerance bands +/- 1
    fig.add_hline(y=1, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-1, line_dash="dash", line_color="grey")

    # Optional: connect points by spot for clarity
    fig.update_traces(
        mode='markers+lines',
        marker=dict(size=12, line=dict(width=2)),
        line=dict(width=1)
    )

    fig.show()

    return


In [32]:
# plotting relative x andy-pos, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last 24 months
plotly_spot_position(sub_df, "rel_xpos", "Gantry 4", "XRV-3000", 70, 0, 24, exclude_centre=True)
plotly_spot_position(sub_df, "rel_ypos", "Gantry 4", "XRV-4000", 70, 0, 24, exclude_centre=True)

## plot another device

In [33]:
plotly_spot_position(sub_df, "rel_xpos", "Gantry 2", "XRV-3000", 70, 0, 24, exclude_centre=True)
plotly_spot_position(sub_df, "rel_ypos", "Gantry 2", "XRV-4000", 70, 0, 24, exclude_centre=True)

In [34]:
start_date = pd.Timestamp.today() - pd.DateOffset(months=12)

selected_df = sub_df[
    (sub_df["MachineName"] == "Gantry 2") &
    (sub_df["Device"] == "XRV-3000") &
    (sub_df['ADate'] >= start_date)
].copy()

# Calculate average relative x-pos per adate and energy
selected_df['avg_rel_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])["rel_xpos"].transform('mean')

selected_df.head(5)

,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,px_pos,py_pos,abs_xpos,abs_ypos,centre_abs_x,centre_abs_y,rel_xpos,rel_ypos,avg_rel_pos
32499,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Bottom-Centre,0.3273,125.5616,0,125,0.3273,0.5616,0.158,0.4836,0.1693,0.0780,0.0922
32500,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Bottom-Left,-124.7349,125.5062,-125,125,0.2651,0.5062,0.158,0.4836,0.1071,0.0226,0.0922
32501,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Bottom-Right,125.6069,125.4004,125,125,0.6069,0.4004,0.158,0.4836,0.4489,-0.0832,0.0922
32502,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Centre,0.1580,0.4836,0,0,0.1580,0.4836,0.158,0.4836,0.0000,0.0000,0.0922
32503,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Left,-124.9901,0.9085,-125,0,0.0099,0.9085,0.158,0.4836,-0.1481,0.4249,0.0922


In [35]:
def plotly_ave_spot_position(df, parameter, gantry, device, n_months, exclude_centre=True):
    """ plot average relative spot position across all spot positions with the same adate and energy
        df = dataframe
        gantry = "Gantry 1", "Gantry 2",
        parameter = "rel_xpos" or "rel_ypos",
        device = "XRV-3000", "XRV-4000"
        n_months = int
     """

    # only show data from last n months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df['ADate'] >= start_date)
    ].copy()

    if exclude_centre:
        selected_df = selected_df[selected_df["Spot"] != "Centre"]

    # Calculate average relative position per adate and energy
    selected_df['avg_rel_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])[parameter].transform('mean')

    # want to display energy as discrete colour not spectrum
    selected_df['Energy'] = selected_df['Energy'].astype(int).astype(str)

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y='avg_rel_pos',
        symbol='Gantry Angle',
        color='Energy',
        title=f'average relative shift wrt Centre: {parameter}',
        labels={'ADate': 'Date', 'avg_rel_pos': 'Average relative shift (mm)'},
        height=500
    )

    # Add tolerance bands +/- 1
    fig.add_hline(y=1, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-1, line_dash="dash", line_color="grey")

    # Optional: connect points by spot for clarity
    fig.update_traces(
        mode='markers+lines',
        marker=dict(size=12, line=dict(width=2)),
        line=dict(width=1)
    )

    fig.show()

    return

In [36]:
plotly_ave_spot_position(sub_df, "rel_xpos", "Gantry 1", "XRV-3000", 24, exclude_centre=True)